In [ ]:
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

# Reload and preprocess
data = sio.loadmat('../data/S17_E1_A1.mat')
emg = data['emg']
labels = data['restimulus']
fs = 2000

def bandpass_filter(signal, lowcut=20, highcut=500, fs=2000, order=4):
    nyq = fs / 2
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return filtfilt(b, a, signal)

# Filter all 12 channels
emg_filtered = np.zeros_like(emg)
for ch in range(12):
    emg_filtered[:, ch] = bandpass_filter(emg[:, ch])

print("Filtered EMG shape:", emg_filtered.shape)

# Feature extraction functions
def mav(window):
    return np.mean(np.abs(window))

def rms(window):
    return np.sqrt(np.mean(window**2))

def zero_crossing_rate(window, threshold=1e-6):
    zc = 0
    for i in range(1, len(window)):
        if ((window[i] >= threshold and window[i-1] < threshold) or
            (window[i] < -threshold and window[i-1] >= -threshold)):
            zc += 1
    return zc

def waveform_length(window):
    return np.sum(np.abs(np.diff(window)))

def extract_features(window):
    features = []
    for ch in range(window.shape[1]):
        ch_signal = window[:, ch]
        features.extend([
            mav(ch_signal),
            rms(ch_signal),
            zero_crossing_rate(ch_signal),
            waveform_length(ch_signal)
        ])
    return np.array(features)

# Sliding window extraction
window_ms = 200
step_ms = 100  # 50% overlap
window_samples = int(window_ms * fs / 1000)
step_samples = int(step_ms * fs / 1000)

X = []  # feature vectors
y = []  # labels

for start in range(0, len(emg_filtered) - window_samples, step_samples):
    end = start + window_samples
    window = emg_filtered[start:end, :]
    label = labels[start + window_samples//2, 0]  # label at window center
    
    X.append(extract_features(window))
    y.append(label)

X = np.array(X)
y = np.array(y)

print("Feature matrix shape:", X.shape)
print("Labels shape:", y.shape)
print("Gesture classes:", np.unique(y))

# Plot MAV (feature 0) for each channel across gesture classes
# to verify features carry discriminative information

fig, axes = plt.subplots(3, 4, figsize=(14, 8))
axes = axes.flatten()

for ch in range(12):
    mav_feature_idx = ch * 4  # MAV is first feature per channel
    
    gesture_means = []
    for gesture in range(18):
        mask = y == gesture
        if mask.sum() > 0:
            gesture_means.append(X[mask, mav_feature_idx].mean())
        else:
            gesture_means.append(0)
    
    axes[ch].bar(range(18), gesture_means)
    axes[ch].set_title(f'Channel {ch+1} MAV')
    axes[ch].set_xlabel('Gesture')
    axes[ch].set_ylabel('MAV')

plt.suptitle('Mean MAV per gesture per channel', fontsize=12)
plt.tight_layout()
plt.show()

Filtered EMG shape: (1804768, 12)
Feature matrix shape: (9022, 48)
Labels shape: (9022,)
Gesture classes: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17]
